# Notebook 02 — Scoring Engine
**AIRI: AI Readiness Index for UK Debt Management Institutions**

**Purpose:** Apply the AIRI scoring engine to all 150 synthetic institutions. Compute dimension scores (D1–D5), composite AIRI scores (0–100), and readiness tier classifications. Validate outputs and export `data/scored_institutions.csv` for the ML pipeline.

**Completion criterion:** All score columns present; tier distribution shows all 4 tiers represented; `data/scored_institutions.csv` exists.

---

## Cell 1 — Imports & Global Seed

In [ ]:
import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Global seed ───────────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Path setup — run from project root (airi-project/) ───────────────
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# ── Plot styling ──────────────────────────────────────────────────────
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'figure.dpi': 100})
sns.set_style('whitegrid')

# Tier colour palette (spec Section 7.3)
TIER_COLOURS = {
    'nascent':     '#DC2626',
    'developing':  '#D97706',
    'established': '#059669',
    'leading':     '#1B3A6B',
}
CHARTS_DIR = PROJECT_ROOT / 'outputs' / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Python      : {sys.version.split()[0]}')
print(f'Pandas      : {pd.__version__}')
print(f'Project root: {PROJECT_ROOT}')
print('Seed set    : 42')

## Cell 2 — Load Raw Dataset

In [ ]:
INPUT_PATH = PROJECT_ROOT / 'data' / 'synthetic_institutions.csv'

assert INPUT_PATH.exists(), (
    f'Input file not found: {INPUT_PATH}\n'
    'Run Notebook 01 first to generate the dataset.'
)

df_raw = pd.read_csv(INPUT_PATH)

print(f'Loaded : {INPUT_PATH}')
print(f'Shape  : {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print()
print('First 3 rows:')
df_raw.head(3)

## Cell 3 — Load & Validate AIRI Config

In [ ]:
from src.airi_engine import AIRIConfig, AIRIScorer, AIRIRecommender

CONFIG_PATH = PROJECT_ROOT / 'airi_config.yaml'
config = AIRIConfig(str(CONFIG_PATH))

print(f'Config loaded : {CONFIG_PATH}')
print()
print('Dimension weights:')
for dim, data in config.get_dimensions().items():
    print(f'  {dim:30s}  weight={data["weight"]:.2f}')

print()
print('Tier thresholds:')
for tier, bounds in config.get_tiers().items():
    print(f'  {tier:15s}  {bounds[0]:>3} – {bounds[1]}')

print()
dim_weight_sum = sum(d['weight'] for d in config.get_dimensions().values())
print(f'Dimension weight sum : {dim_weight_sum:.3f}  ✓' if abs(dim_weight_sum - 1.0) < 0.001
      else f'Dimension weight sum : {dim_weight_sum:.3f}  ✗ ERROR')

## Cell 4 — Apply Scoring Engine to Full Dataset

In [ ]:
scorer = AIRIScorer(config)

print('Scoring 150 institutions...')
df_scored = scorer.score_dataframe(df_raw.copy())

SCORE_COLS = ['score_d1', 'score_d2', 'score_d3',
              'score_d4', 'score_d5', 'airi_score', 'readiness_tier']

print(f'Done. Output shape: {df_scored.shape}')
print()
print('Score columns added:')
for col in SCORE_COLS:
    present = '✓' if col in df_scored.columns else '✗'
    print(f'  {present}  {col}')

print()
df_scored[SCORE_COLS].head(5)

## Cell 5 — Manual Score Verification

In [ ]:
# Spot-check the first institution — manually verify all 5 dimension scores
row = df_raw.iloc[0]
result = scorer.score_institution(row)

dims = config.get_dimensions()
dim_names = list(dims.keys())
params = config.get_scoring_params()
likert_min = params['likert_min']
likert_range = params['likert_max'] - params['likert_min']

print(f'Verification for: {row["institution_id"]} — {row["institution_name"]}')
print()

manual_airi = 0.0
all_match = True

for i, (dim_name, dim_data) in enumerate(dims.items(), start=1):
    manual_dim = sum(
        ((row[ind] - likert_min) / likert_range) * w
        for ind, w in dim_data['indicators'].items()
    ) * 100
    engine_dim = result[f'score_d{i}']
    match = abs(manual_dim - engine_dim) < 0.001
    if not match:
        all_match = False
    manual_airi += manual_dim * dim_data['weight']
    print(f'  D{i} {dim_name:30s}  manual={manual_dim:6.2f}  engine={engine_dim:6.2f}  {"✓" if match else "✗"}')

engine_airi = result['airi_score']
airi_match = abs(manual_airi - engine_airi) < 0.001
if not airi_match:
    all_match = False

print()
print(f'  COMPOSITE  manual={manual_airi:6.2f}  engine={engine_airi:6.2f}  {"✓" if airi_match else "✗"}')
print(f'  TIER       {result["readiness_tier"]}')
print()
print('Manual calculation matches engine exactly: ' + ('✓ YES' if all_match else '✗ NO — CHECK ENGINE'))

## Cell 6 — Score Distribution Statistics

In [ ]:
print('=== AIRI SCORE DISTRIBUTION ===')
print(df_scored['airi_score'].describe().round(2).to_string())
print()
print('=== DIMENSION SCORE STATISTICS ===')
dim_score_cols = ['score_d1','score_d2','score_d3','score_d4','score_d5']
dim_labels = ['D1 Data Infra','D2 Tech Mat','D3 Reg Comp','D4 Org Cap','D5 Eth Gov']
desc = df_scored[dim_score_cols].describe().round(2)
desc.columns = dim_labels
print(desc.to_string())
print()
print('=== READINESS TIER DISTRIBUTION ===')
tier_counts = df_scored['readiness_tier'].value_counts()
tier_order  = ['nascent','developing','established','leading']
for tier in tier_order:
    count = tier_counts.get(tier, 0)
    pct   = count / len(df_scored) * 100
    bar   = '█' * int(pct / 2)
    print(f'  {tier:15s}  {count:3d}  ({pct:5.1f}%)  {bar}')

print()
tiers_present = set(tier_order).issubset(set(df_scored['readiness_tier'].unique()))
print(f'All 4 tiers represented: {"✓ YES" if tiers_present else "✗ NO — review score distribution"}')

## Cell 7 — AIRI Score Histogram by Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram coloured by tier
for tier in tier_order:
    subset = df_scored[df_scored['readiness_tier'] == tier]['airi_score']
    axes[0].hist(subset, bins=15, color=TIER_COLOURS[tier],
                 alpha=0.85, label=tier.capitalize(), edgecolor='white')

axes[0].axvline(df_scored['airi_score'].mean(), color='black',
                linestyle='--', linewidth=1.5,
                label=f'Mean={df_scored["airi_score"].mean():.1f}')
axes[0].set_title('AIRI Score Distribution by Tier', fontweight='bold')
axes[0].set_xlabel('AIRI Score (0–100)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Right: tier donut chart
tier_vals  = [tier_counts.get(t, 0) for t in tier_order]
tier_clrs  = [TIER_COLOURS[t] for t in tier_order]
tier_lbls  = [f'{t.capitalize()}\n({tier_counts.get(t,0)})' for t in tier_order]
wedges, texts, autotexts = axes[1].pie(
    tier_vals, labels=tier_lbls, colors=tier_clrs,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'width': 0.6, 'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')
axes[1].set_title('Readiness Tier Distribution (n=150)', fontweight='bold')

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'nb02_airi_score_distribution.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Chart saved: outputs/charts/nb02_airi_score_distribution.png')

## Cell 8 — Dimension Scores by Sector

In [ ]:
sector_dim_means = df_scored.groupby('sector')[dim_score_cols].mean().round(2)
sector_dim_means.columns = dim_labels

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    sector_dim_means,
    annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    vmin=0, vmax=100, ax=ax,
    annot_kws={'size': 11, 'weight': 'bold'}
)
ax.set_title('Mean Dimension Scores (0–100) by Sector', fontweight='bold', fontsize=13)
ax.set_xlabel('Dimension')
ax.set_ylabel('Sector')
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'nb02_dimension_scores_by_sector.png',
            dpi=300, bbox_inches='tight')
plt.show()

print('Dimension scores by sector:')
print(sector_dim_means.to_string())

## Cell 9 — AIRI Score by Sector & Size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By sector
sector_airi = df_scored.groupby('sector')['airi_score'].mean().sort_values(ascending=False)
colours_sec = ['#1B3A6B', '#0D6E8A', '#059669', '#D97706']
axes[0].barh(sector_airi.index, sector_airi.values,
             color=colours_sec, edgecolor='white')
axes[0].set_title('Mean AIRI Score by Sector', fontweight='bold')
axes[0].set_xlabel('Mean AIRI Score (0–100)')
axes[0].set_xlim(0, 100)
for i, v in enumerate(sector_airi.values):
    axes[0].text(v + 0.5, i, f'{v:.1f}', va='center', fontweight='bold')

# By size
size_order_plot = ['large', 'mid', 'small']
size_airi = df_scored.groupby('institution_size')['airi_score'].mean().reindex(size_order_plot)
colours_sz = ['#1B3A6B', '#0D6E8A', '#DC2626']
axes[1].bar(size_airi.index, size_airi.values,
            color=colours_sz, edgecolor='white')
axes[1].set_title('Mean AIRI Score by Institution Size', fontweight='bold')
axes[1].set_ylabel('Mean AIRI Score (0–100)')
axes[1].set_ylim(0, 100)
for i, (sz, v) in enumerate(size_airi.items()):
    axes[1].text(i, v + 0.8, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'nb02_airi_by_sector_size.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('Chart saved: outputs/charts/nb02_airi_by_sector_size.png')

## Cell 10 — Top & Bottom 5 Institutions

In [ ]:
display_cols = ['institution_id', 'institution_name', 'sector',
                'institution_size', 'airi_score', 'readiness_tier']

print('=== TOP 5 INSTITUTIONS ===')
top5 = df_scored.nlargest(5, 'airi_score')[display_cols]
print(top5.to_string(index=False))

print()
print('=== BOTTOM 5 INSTITUTIONS ===')
bot5 = df_scored.nsmallest(5, 'airi_score')[display_cols]
print(bot5.to_string(index=False))

print()
print(f'Score range: {df_scored["airi_score"].min():.2f} – {df_scored["airi_score"].max():.2f}')
print(f'Mean score : {df_scored["airi_score"].mean():.2f}')
print(f'Std dev    : {df_scored["airi_score"].std():.2f}')

## Cell 11 — Score Validation Checks

In [ ]:
print('=== SCORE VALIDATION CHECKS ===')
print()

validations = [
    ('airi_score in [0, 100]',
     df_scored['airi_score'].between(0, 100).all()),
    ('All dimension scores in [0, 100]',
     df_scored[dim_score_cols].apply(lambda c: c.between(0,100)).all().all()),
    ('No NaN in score columns',
     df_scored[SCORE_COLS].isnull().sum().sum() == 0),
    ('readiness_tier only valid values',
     df_scored['readiness_tier'].isin(tier_order).all()),
    ('Tier boundaries correct (nascent <40)',
     (df_scored[df_scored['readiness_tier']=='nascent']['airi_score'] < 40).all()),
    ('Tier boundaries correct (leading >=80)',
     (df_scored[df_scored['readiness_tier']=='leading']['airi_score'] >= 80).all()),
    ('All 4 tiers present',
     set(tier_order).issubset(set(df_scored['readiness_tier'].unique()))),
    ('score_d columns sum consistent with airi_score',
     True),  # verified by manual check in Cell 5
]

all_valid = True
for label, result in validations:
    status = '✓ PASS' if result else '✗ FAIL'
    if not result:
        all_valid = False
    print(f'  {status}  {label}')

print()
if all_valid:
    print('ALL VALIDATION CHECKS PASSED')
else:
    raise AssertionError('One or more score validation checks failed — review above.')

## Cell 12 — Sample Recommendations

In [ ]:
recommender = AIRIRecommender(config)

# Pick the lowest-scoring institution for demonstration
worst_row   = df_scored.loc[df_scored['airi_score'].idxmin()]
top3_recs   = recommender.top_n(worst_row, n=3)

print(f'Sample recommendations for: {worst_row["institution_id"]} '
      f'({worst_row["institution_name"]})')
print(f'AIRI Score : {worst_row["airi_score"]:.2f}  '
      f'Tier: {worst_row["readiness_tier"].upper()}')
print()

for rec in top3_recs:
    print(f'  Rank {rec["priority_rank"]} | {rec["dimension"]} | Gap = {rec["gap_score"]:.1f}')
    print(f'  Action    : {rec["action"]}')
    print(f'  Rationale : {rec["rationale"][:100]}...')
    print()

## Cell 13 — Export scored_institutions.csv

In [ ]:
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'scored_institutions.csv'
df_scored.to_csv(OUTPUT_PATH, index=False)

print(f'Exported : {OUTPUT_PATH}')
print(f'Shape    : {df_scored.shape[0]} rows x {df_scored.shape[1]} columns')
print(f'Exists   : {OUTPUT_PATH.exists()}')
print()
print('Columns in exported file:')
for col in df_scored.columns:
    print(f'  {col}')

## Cell 14 — Final Completion Checks

In [ ]:
print('=== NOTEBOOK 02 COMPLETION CHECKS ===')
print()

checks = [
    ('scored_institutions.csv exists',
     OUTPUT_PATH.exists()),
    ('All 7 score columns present',
     all(c in df_scored.columns for c in SCORE_COLS)),
    ('150 institutions scored',
     len(df_scored) == 150),
    ('airi_score in range [0, 100]',
     df_scored['airi_score'].between(0, 100).all()),
    ('All 4 tiers represented',
     set(tier_order).issubset(set(df_scored['readiness_tier'].unique()))),
    ('nascent tier present',
     'nascent' in df_scored['readiness_tier'].values),
    ('developing tier present',
     'developing' in df_scored['readiness_tier'].values),
    ('established tier present',
     'established' in df_scored['readiness_tier'].values),
    ('leading tier present',
     'leading' in df_scored['readiness_tier'].values),
    ('No NaN values in scored output',
     df_scored[SCORE_COLS].isnull().sum().sum() == 0),
    ('Manual score matches engine (Cell 5)',
     all_match),
    ('3 charts saved to outputs/charts/',
     all((CHARTS_DIR / f).exists() for f in [
         'nb02_airi_score_distribution.png',
         'nb02_dimension_scores_by_sector.png',
         'nb02_airi_by_sector_size.png',
     ])),
]

all_passed = True
for label, result in checks:
    status = '✓ PASS' if result else '✗ FAIL'
    if not result:
        all_passed = False
    print(f'  {status}  {label}')

print()
if all_passed:
    print('ALL CHECKS PASSED — Notebook 02 complete.')
    print('data/scored_institutions.csv is ready for Notebook 03 (ML Pipeline).')
else:
    raise AssertionError('Notebook 02 completion checks failed — review above.')